# Module 2 · Lesson 04: The PCTF Framework

**PCTF = Persona · Context · Task · Format**

A systematic approach to writing effective prompts. Instead of guessing,
use this framework to structure every prompt.

## What you will learn
1. The four PCTF components
2. Real-world examples (code review, documentation)
3. Reusable **prompt templates**
4. Adding **negative constraints**
5. Building an interactive PCTF builder

In [1]:
# ── Setup ──────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown
 
load_dotenv(Path.cwd().parent / ".env")
 
from openai import OpenAI
client = OpenAI()
 
def ask(prompt, system=None, temperature=0.7, max_tokens=500):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(
        model="gpt-4o-mini", messages=msgs,
        temperature=temperature, max_tokens=max_tokens
    )
    return r.choices[0].message.content
 
print("✅ Ready")

✅ Ready


---
## The PCTF Framework

| Component | Question | Example |
|-----------|----------|---------|
| **P**ersona | Who should the AI be? | "You are a senior code reviewer" |
| **C**ontext | What's the situation? | "We're building a FastAPI backend" |
| **T**ask | What should it do? | "Review this function for bugs" |
| **F**ormat | How should it respond? | "Use a numbered list with severity" |

---
## 1. Example: Code Review

In [3]:
system_prompt = """You are senior Python developer with 10 years of experience.
You specialize in code quality, security and performance.
"""
 
user_prompt = """Context: We' re building a REST API with FastAPI. This function handles user login.
 
Task: Review this code for bugs, security issues and performance problems.
 
```python
def login(username, password):
    user = db.query(f"SELECT * FROM users WHERE username='{username}'")
    if user and user.password == password:
        token = str(random.randint(1000, 9999))
        return {"token": token}
    return {"error": "Invalid credentials"}
```
 
Format: For each issue found, provide:
1. Issue severity (Critical, Warning, Suggestion)
2. Line reference
3. Problem description
4. Corrected code
 
"""
 
result = ask(user_prompt, system=system_prompt)
display(Markdown(result))
 

Here’s a review of the provided `login` function, highlighting issues related to bugs, security, and performance:

### Issue 1: SQL Injection Vulnerability
1. **Severity**: Critical
2. **Line Reference**: `user = db.query(f"SELECT * FROM users WHERE username='{username}'")`
3. **Problem Description**: The code constructs an SQL query using string interpolation, which is vulnerable to SQL injection attacks. An attacker could manipulate the `username` input to execute arbitrary SQL commands.
4. **Corrected Code**:
   ```python
   user = db.query("SELECT * FROM users WHERE username = :username", {"username": username})
   ```

### Issue 2: Password Storage and Comparison
1. **Severity**: Critical
2. **Line Reference**: `if user and user.password == password:`
3. **Problem Description**: The password is being compared in plain text. Passwords should never be stored or compared in plain text; instead, they should be hashed using a strong hashing algorithm (e.g., bcrypt) and compared using a secure method.
4. **Corrected Code**:
   ```python
   from passlib.context import CryptContext

   pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

   # Assuming user.password is a hashed password
   if user and pwd_context.verify(password, user.password):
   ```

### Issue 3: Token Generation
1. **Severity**: Warning
2. **Line Reference**: `token = str(random.randint(1000, 9999))`
3. **Problem Description**: The token generation method is insecure, as it uses `random.randint`, which is not cryptographically secure. This could allow attackers to easily guess valid tokens.
4. **Corrected Code**:
   ```python
   import secrets

   token = secrets.token_hex(16)  # Generates a secure random token
   ```

### Issue 4: Return Consistency
1. **Severity**: Suggestion
2. **Line Reference**: `return {"error": "Invalid credentials"}`
3. **Problem Description**: The return structure for invalid credentials is inconsistent with the successful response. This could lead to confusion in client-side handling. It's better to maintain a consistent response format.
4. **Corrected Code**:
   ```python
   return {"error": "Invalid credentials"}, 401

---
## 2. Example: API Documentation

In [4]:
system_prompt = """You are a technical writer specializing in API documentation.
Write clear, concise docs that developers can use immediately.
"""
 
user_prompt = """Context: We have a user management API built with FastAPI.
 
Task: Write API documentation for this endpoint.
 
```python
@app.post("/users")
def create_user(name: str, email: str, role: str = "viewer"):
    user = User(name=name, email=email, role=role)
    db.add(user)
    return {"id": user.id, "name": user.name}
```
 
Format: Include:
- Endpoint summary
- Parameters table (name, type, required, description)
- Example request (curl)
- Example response (JSON)
- Possible error codes"""
 
result = ask(user_prompt, system=system_prompt, max_tokens=1600)
display(Markdown(result))

# User Management API Documentation

## Endpoint Summary
**POST /users**  
This endpoint creates a new user in the system with a specified name, email, and role. If the role is not provided, it defaults to "viewer".

---

## Parameters

| Name  | Type   | Required | Description                           |
|-------|--------|----------|---------------------------------------|
| name  | string | Yes      | The full name of the user.           |
| email | string | Yes      | The email address of the user.       |
| role  | string | No       | The role of the user (default: "viewer"). Valid roles include "viewer", "editor", and "admin". |

---

## Example Request

```bash
curl -X POST http://yourapi.com/users \
-H "Content-Type: application/json" \
-d '{
  "name": "John Doe",
  "email": "john.doe@example.com",
  "role": "editor"
}'
```

---

## Example Response

### Success (HTTP Status 200)

```json
{
  "id": 1,
  "name": "John Doe"
}
```

### Error Response (HTTP Status 400)

```json
{
  "detail": "Email already in use."
}
```

### Error Response (HTTP Status 422)

```json
{
  "detail": [
    {
      "loc": ["body", "email"],
      "msg": "value is not a valid email address",
      "type": "value_error.email"
    }
  ]
}
```

---

## Possible Error Codes

- **400 Bad Request**: The request was invalid. This can occur if the email is already in use or if required fields are missing.
- **422 Unprocessable Entity**: The request was well-formed but contained semantic errors, such as an invalid email format.

Make sure to handle these error responses appropriately in your application.

---
## 3. Reusable Prompt Templates

In production, you build **template functions** for common tasks:

In [5]:
def create_pctf_prompt(persona: str, context: str, task: str, format_spec: str) -> str:
    """Build a structured PCTF prompt"""
    return f"""## Persona
{persona}
 
## Context
{context}
 
## Task
{task}
 
## Format
{format_spec}"""

In [6]:
prompt = create_pctf_prompt(
    persona="You are a database expert with deep knowledge of SQL optimization.",
    context="We have a PostgreSQL database with 10M rows in the `orders` table. Queries are slow.",
    task="Suggest 3 specific optimizations for this query: SELECT * FROM orders WHERE status='pending' AND created_at > NOW() - INTERVAL '7 days' ORDER BY total DESC",
    format_spec="For each optimization: title, explanation, SQL example, expected improvement."
)
 
result = ask(prompt, max_tokens=600)
display(Markdown(result))

### Optimization 1: Create Index on Status and Created_at
**Explanation:** Creating a composite index on the `status` and `created_at` columns will significantly speed up the filtering process. PostgreSQL can quickly locate rows that match the criteria, reducing the search space for the `ORDER BY` clause.

**SQL Example:**
```sql
CREATE INDEX idx_orders_status_created_at ON orders (status, created_at);
```

**Expected Improvement:** This optimization can reduce the query execution time dramatically (potentially by 70-90%) by allowing PostgreSQL to efficiently filter the rows before applying the ordering.

---

### Optimization 2: Use a Covering Index
**Explanation:** A covering index that includes the `total` column will allow the query to retrieve all necessary data from the index itself, rather than needing to access the table rows. This can improve performance, especially for large tables, as it reduces I/O operations.

**SQL Example:**
```sql
CREATE INDEX idx_orders_status_created_at_total ON orders (status, created_at) INCLUDE (total);
```

**Expected Improvement:** This could lead to a performance improvement of around 50-80%, as the database can fetch results directly from the index without needing to read the full rows from the table.

---

### Optimization 3: Partitioning the Table
**Explanation:** If the `orders` table grows significantly, consider partitioning it based on a relevant column, such as `created_at`. This can help in managing large datasets more effectively and improve query performance by limiting the number of rows scanned.

**SQL Example:**
```sql
CREATE TABLE orders_y2023 PARTITION OF orders 
FOR VALUES FROM ('2023-01-01') TO ('2023-12-31');
```
(Note: Actual partitioning would depend on your data distribution and access patterns.)

**Expected Improvement:** Depending on the partitioning strategy and access patterns, this can yield improvements of 30-70% by reducing the number of rows scanned during the query execution.

--- 

These optimizations should be evaluated in context of the specific workload and access patterns of your application for best results.

---
## 4. Negative Constraints

Telling the model what **NOT** to do is as important as what to do:

In [7]:
prompt = create_pctf_prompt(
    persona="You are senior developer mentoring a junior",
    context="A junior developer submitted a pull request with this code.",
    task="""Review this code:
```python
data = []
for i in range(len(items)):
    if items[i]['active'] == True:
        data.append(items[i]['name'].upper())
```""",
    format_spec="""Provide feedback. DO NOT:
- Be condescending or harsh
- Rewrite the entire function
- Use jargon without explanation
 
DO:
- Explain WHY each change improves the code
- Show improved version
- Encourage what they did right
"""
)

In [8]:
result = ask(prompt, max_tokens=1000)
display(Markdown(result))

Thanks for submitting your pull request! You've made a good start by filtering active items and transforming their names to uppercase. I appreciate your efforts! However, there are a few improvements we can make to enhance the code's readability and efficiency.

### Feedback on the Current Code
1. **Using List Comprehensions**: Python has a feature called list comprehensions that allows you to create lists in a more concise and readable manner. This can make your code cleaner and easier to understand at a glance.

2. **Boolean Comparison**: When checking if a value is `True`, you don't need to explicitly compare it with `== True`. In Python, you can directly use the condition itself, which makes the code simpler.

3. **Avoiding `range(len(...))`**: Instead of using `range(len(items))` to iterate over indices, you can directly loop through the items themselves. This eliminates the need to access the list using indices and makes the code cleaner.

### Suggested Improvements
Here’s an improved version of your code with these suggestions implemented:

```python
data = [item['name'].upper() for item in items if item['active']]
```

### Explanation of Changes
- **List Comprehension**: This change makes the code shorter and clearer. It shows that you're creating a new list in one line, which is easier to read than the traditional loop.
- **Simplified Condition**: Using `if item['active']` directly checks the truthiness of the value, making the code more Pythonic.
- **Direct Iteration**: By iterating directly over `items`, you avoid the potential for errors that come from managing indices and improve the clarity of what you're working with.

### Encouragement
You've laid a solid foundation by filtering and transforming the data. Adopting these improvements will not only enhance your code's readability but also make it more efficient. Keep up the great work, and continue exploring Python's features! If you have any questions or want to discuss further, feel free to ask.

---
## 5. Exercise — Build Your Own PCTF Prompt 🏋️

---
## Key Takeaways 📝

| Component | Tips |
|-----------|------|
| **Persona** | Be specific about expertise level and domain |
| **Context** | Include tech stack, constraints, audience |
| **Task** | One clear, specific action verb |
| **Format** | Tables, lists, JSON — be explicit |
| **Negative constraints** | Say what NOT to do to avoid common pitfalls |
| **Templates** | Build reusable functions for common tasks |

---
**Next:** `05_prompt_evaluation.ipynb` — Measure and compare prompt effectiveness